In [ ]:
# py -m pip install git+https://github.com/pbtamarona/OpenPyTEA
from openpytea.equipment import *
from openpytea.plant import *
from openpytea.analysis import *
from openpytea.plotting import *

In [ ]:
#OPEX Input
country, region = 'Netherlands', 'None'
interest_rate, project_lifetime, plant_utilization = 0.09, 20, 0.95
h2_prod = 100_000  # Daily H₂ production in kg (100 tonnes/day)
h2_price = 5       # US$/kg
tax_rate = 0.258

operator_rate, operator_std, operator_max, operator_min = 39.04, 12.31, 55, 23.43
# Operator hourly rate for USA-Gulf Coast, in US$/h


elecs_price, elecs_std, elecs_max, elecs_min = 0.125, 0.05, 0.5, 0.05
# Price of electricity, in US$/kWh

nat_gas_price, nat_gas_std, nat_gas_max, nat_gas_min = 0.8646, 0.4, 4.884, 0.30261
# Price of natural gas, in US$/kg

cool_water_price, cool_water_std, cool_water_max, cool_water_min = 2.4592e-4, 1e-4, 4e-4, 1e-5
# Price of cooling water, in US$/MJ

co2_tax, co2_std, co2_max, co2_min = 0.10201, 0.025, 0.19, 0 #US$/kgCO2

In [ ]:
# Original and new plant capacity
capacity_original = 50   #TPD
capacity_new = 100

scale_exponent = 0.6

# Original purchased costs - 2020
stack_cost_2020 = 17_194_000
mechanical_bop_cost_2020 = 11_019_000
electrical_bop_cost_2020 = 16_974_000

# Six-tenths rule
scale_factor = (capacity_new / capacity_original) ** scale_exponent

stack_cost_scaled = stack_cost_2020 * scale_factor
mechanical_bop_cost_scaled = mechanical_bop_cost_2020 * scale_factor
electrical_bop_cost_scaled = electrical_bop_cost_2020 * scale_factor


# Equipment definition
electrolyzer_stack = Equipment("Stack", 0, "Fluids", "Electrolysis", "Stack", "Carbon steel", purchased_cost=stack_cost_scaled, cost_year=2020)

mechanical_bop = Equipment("Mechanical BOP", 0, "Fluids", "Electrolysis", "Mechanical", "Carbon steel", purchased_cost=mechanical_bop_cost_scaled, cost_year=2020)

electrical_bop = Equipment("Electrical BOP", 0, "Electrical", "Electrolysis", "Electrical",   "Carbon steel", purchased_cost=electrical_bop_cost_scaled, cost_year=2020)

we_equipment = [
    electrolyzer_stack,
    mechanical_bop,
    electrical_bop
]


In [ ]:
# Configuration for the WE plant
config = {
    'plant_name'          : 'Water Electrolysis',
    'process_type'        : 'Fluids',
    'country'             : country,
    'region'              : region,
    'equipment'           : we_equipment,
    'interest_rate'       : interest_rate,
    'project_lifetime'    : project_lifetime,
    'plant_utilization'   : plant_utilization,
    'tax_rate'            : tax_rate,
    'operator_hourly_rate': {
        'rate': operator_rate, 'rate_uncertainty': {'std': operator_std, 'max': operator_max, 'min': operator_min}
    },
    'plant_products': {
        'hydrogen': {
            'production': h2_prod,   # Daily production in kg/day
            'price'     : h2_price,  # Price of hydrogen, in US$/kg
        },
    },
    'variable_opex_inputs': {
        'electricity': {
            'consumption': (98*1000)*24, # Electricity consumption in kWh/day (this value comes from the DOE report)
            'price': elecs_price, 'price_uncertainty': {'std': elecs_std, 'max': elecs_max, 'min': elecs_min}
        },
        'water': {
            'consumption': 3.78*2*50_000, # Water consumption in kg/day (this value comes from the DOE report)
            'price': 0.003, 'price_uncertainty': {'std': 0.001, 'max': 0.005, 'min': 0}
        },
    },
}
we_plant = Plant(config)

In [ ]:
# Here, we hardcode the number of fluid and solid processes, instead of letting the Plant object count them automatically
#Assumed no equipment in parallel despite the increase of plant capacity, size of equipment scales with capacity
operators_hired = we_plant.calculate_operators_hired(no_fluid_process=11, no_solid_process=0)
print(f'Estimated # of operators hired for the electrolysis plant: {operators_hired}')

we_plant.update_configuration({'operators_hired': operators_hired})

In [ ]:
#10 years of electrolyser lifetime over a 20 years project
we_plant.update_configuration({'additional_capex_cost': [electrolyzer_stack.purchased_cost],
                               'additional_capex_years': [10]})
# Let's check this by printing the cash flow table
we_plant.calculate_cash_flow(print_results=True)

In [ ]:
we_plant.calculate_fixed_capital(print_results=True, additional_capex=True)   # Calculate the FCI and print the details, including additional CAPEX
we_plant.calculate_variable_opex(print_results=True)   # Calculate the Variable OPEX and print the details
we_plant.calculate_levelized_cost(print_results=True)  # Calculate the LCOH and print the details

In [ ]:
#MONTE CARLO

#Samples
import matplotlib.pyplot as plt

mc_we = monte_carlo(
    we_plant,
    num_samples=1_000_000,
    batch_size=10_000,
    random_seed=12345
)

#Plot
with plt.style.context("default"):

    fig, ax = plot_monte_carlo(
        mc_we,
        metric="LCOP",
        bins=60,
        figsize=(8, 5),
        show=False
    )

    fig.set_dpi(120)
    fig.set_facecolor("white")
    ax.set_facecolor("white")

    ax.set_title("Monte Carlo LCOH - Water Electrolysis", fontsize=14, pad=12)

    ax.set_xlabel(r"LCOH / [US\$ $\cdot$ kg$^{-1}$]", fontsize=13, labelpad=10)
    ax.set_ylabel("Probability density", fontsize=13, labelpad=8)

    ax.tick_params(
        axis="both", which="major", direction="in",
        top=True, right=True, labelsize=11, length=5
    )

    ax.tick_params(
        axis="both", which="minor", direction="in",
        top=True, right=True, length=3
    )

    ax.minorticks_on()

    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(loc="upper right", fontsize=11, frameon=True)

    fig.tight_layout()
    plt.show()

In [ ]:
# Save Monte Carlo results - Water Electrolysis
import json
from pathlib import Path

# Find project root
try:
    start_dir = Path(__file__).resolve().parent
except NameError:
    start_dir = Path.cwd()

project_root = next(
    (p for p in [start_dir, *start_dir.parents]
     if all((p / folder).is_dir() for folder in ["SMR", "WaterElectrolysis"])),
    None
)

if project_root is None:
    raise FileNotFoundError("Project root not found.")

# Output directory
results_dir = project_root / "WaterElectrolysis"
results_dir.mkdir(parents=True, exist_ok=True)

mc_file = results_dir / "LCOH_MonteCarlo_WE.json"

# Prepare Monte Carlo results
mc_export = {
    "name": "WE",
    "metrics": {
        "LCOP": mc_we["metrics"]["LCOP"].tolist()
    }
}

# Save JSON
with open(mc_file, "w") as f:
    json.dump(mc_export, f, indent=4)

print(f"JSON saved in: {mc_file.resolve()}")